# 5. Loop dmr enrichment

Part of the **[Fig. 5 chapter](fig5.md)** — see it for the panel-by-panel map, run order, and data inputs. The first code cell sets `ENTEX_ROOT` and activates the no-overwrite guard (see the [Reproduction guide](../reproduce.md)).


In [ ]:
# === Reproduction setup — edit ENTEX_ROOT / REF_ROOT for your machine ===
# All absolute paths below resolve from these two roots; the working directory
# is the original analysis/ folder, and repro_guard prevents any existing file
# from being overwritten when you re-run this notebook. See the book's
# 'Reproduction guide' chapter for details.
import os, sys
ENTEX_ROOT = os.environ.get('ENTEX_ROOT', '/large_storage/zhoulab/zhoujt/project/ENTEx')
REF_ROOT   = os.environ.get('REF_ROOT',   '/large_storage/zhoulab/ref')
BOOK_ROOT  = os.environ.get('BOOK_ROOT',  f'{ENTEX_ROOT}/analysis/HumanCellEpigenomeAtlas')
sys.path.insert(0, BOOK_ROOT)
os.chdir(f'{ENTEX_ROOT}/analysis')   # original working directory
import repro_guard                    # no-overwrite guard (default: skip existing)


In [ ]:
cd /large_storage/zhoulab/zhoujt/project/ENTEx/

for i in `seq 0 35`; do
if test -e "DMR/majortype-subtype/c${i}_dmr.bed"; then 
awk '{printf("%s\t%d\t%d\n%s\t%d\t%d\n",$1,$2,$3,$4,$5,$6)}' loop/majortype/c${i}/c${i}.loop_summit.bedpe | sort -k1,1 -k2,2n -u > tmp.txt;
echo c${i} $(bedtools intersect -wa -a DMR/majortype-subtype/c${i}_dmr.bed -b tmp.txt | sort -k1,1 -k2,2n -u | wc -l) $(cat DMR/majortype-subtype/c${i}_dmr.bed | wc -l) $(cat tmp.txt | wc -l);
fi;
done | awk '{printf("%s\t%d\t%d\t%d\n",$1,$2,$3,$4)}' > analysis/loop_dmr_enrich/subtype_dmr_loop_summit.txt

for i in `seq 0 35`; do
if test -e "DMR/majortype-subtype/c${i}_dmr.bed"; then 
awk '{printf("%s\t%d\t%d\n%s\t%d\t%d\n",$1,$2,$3,$4,$5,$6)}' loop/majortype/c${i}/c${i}.loop.bedpe | sort -k1,1 -k2,2n -u > tmp.txt;
echo c${i} $(bedtools intersect -wa -a DMR/majortype-subtype/c${i}_dmr.bed -b tmp.txt | sort -k1,1 -k2,2n -u | wc -l) $(cat DMR/majortype-subtype/c${i}_dmr.bed | wc -l) $(cat tmp.txt | wc -l);
fi;
done | awk '{printf("%s\t%d\t%d\t%d\n",$1,$2,$3,$4)}' > analysis/loop_dmr_enrich/subtype_dmr_loop.txt


In [ ]:
bedtools merge -i /large_storage/zhoulab/ref/blacklist/hg38_bismark_loop_blacklist.bed | bedtools slop -i - -g /large_storage/zhoulab/ref/hg38/fasta/hg38.chrom.sizes -b 70000 | awk '$1!="chrX"' | bedtools intersect -wa -a /large_storage/zhoulab/ref/hg38/fasta/hg38.main.10kbin.bed -b - -v | sort -k1,1 -k2,2n -u | awk '($1!="chrX") && ($1!="chrY")' > /large_storage/zhoulab/ref/hg38/fasta/hg38.auto.anchor.bed


In [ ]:
for i in `seq 0 35`; do
if test -e "DMR/majortype-subtype/c${i}_dmr.bed"; then 
awk '{printf("%s\t%d\t%d\n%s\t%d\t%d\n",$1,$2,$3,$4,$5,$6)}' loop/majortype/c${i}/c${i}.loop_summit.bedpe | sort -k1,1 -k2,2n -u > tmp.txt;
echo c${i} $(bedtools intersect -wa -a tmp.txt -b DMR/majortype-subtype/c${i}_dmr.bed | sort -k1,1 -k2,2n -u | wc -l) $(bedtools intersect -wa -a /large_storage/zhoulab/ref/hg38/fasta/hg38.auto.anchor.bed -b DMR/majortype-subtype/c${i}_dmr.bed | sort -k1,1 -k2,2n -u | wc -l) $(cat tmp.txt | wc -l);
fi;
done | awk '{printf("%s\t%d\t%d\t%d\n",$1,$2,$3,$4)}' > analysis/loop_dmr_enrich/loop_summit_subtype_dmr.txt


In [ ]:
for i in `seq 0 35`; do
if test -e "DMR/majortype-subtype/c${i}_dmr.bed"; then 
awk '{printf("%s\t%d\t%d\n%s\t%d\t%d\n",$1,$2,$3,$4,$5,$6)}' loop/majortype/c${i}/c${i}.loop.bedpe | sort -k1,1 -k2,2n -u > tmp.txt;
echo c${i} $(bedtools intersect -wa -a tmp.txt -b DMR/majortype-subtype/c${i}_dmr.bed | sort -k1,1 -k2,2n -u | wc -l) $(bedtools intersect -wa -a /large_storage/zhoulab/ref/hg38/fasta/hg38.auto.anchor.bed -b DMR/majortype-subtype/c${i}_dmr.bed | sort -k1,1 -k2,2n -u | wc -l) $(cat tmp.txt | wc -l);
fi;
done | awk '{printf("%s\t%d\t%d\t%d\n",$1,$2,$3,$4)}' > analysis/loop_dmr_enrich/loop_subtype_dmr.txt


In [ ]:
import os
import numpy as np
import pandas as pd
from glob import glob
from scipy.sparse import csr_matrix
from scipy.stats import zscore
from concurrent.futures import ProcessPoolExecutor, as_completed

import anndata
from ALLCools.mcds import MCDS, RegionDS
from ALLCools.clustering import *
from ALLCools.plot import *

import seaborn as sns
import matplotlib as mpl
import matplotlib.pyplot as plt

mpl.style.use('default')
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42
mpl.rcParams['font.family'] = 'sans-serif'
mpl.rcParams['font.sans-serif'] = 'Helvetica'

import warnings
warnings.filterwarnings("ignore")


In [ ]:
indir = f'{ENTEX_ROOT}/'
outdir = f'{indir}analysis/loop_dmr_enrich/'


In [ ]:
L1_meta = pd.read_csv(f'{indir}L1color.tsv', sep='\t', header=0, index_col=0)
# L1_meta = L1_meta.drop(['c35', 'c36'], axis=0)
L1_meta = L1_meta.drop(['c7'], axis=0)
L1_annot = L1_meta['L1_abbr'].to_dict()
L1_color = L1_meta['color'].to_dict()


In [ ]:
import cooler
chrom_size_path = f'{REF_ROOT}/hg38/fasta/hg38.main.chrom.sizes'
chrom_sizes = cooler.read_chromsizes(chrom_size_path, all_names=True)
chrom_sizes = chrom_sizes.iloc[:22]


In [ ]:
tot_bin = 249871


In [ ]:
data1 = pd.read_csv(f'{outdir}loop_summit_subtype_dmr.txt', sep='\t', header=None, index_col=0, names=['L1','dmrsummit','dmrbin','summit'])
data2 = pd.read_csv(f'{outdir}loop_subtype_dmr.txt', sep='\t', header=None, index_col=0, names=['L1','dmrloop','dmrbin','loop'])
data = pd.concat([data1, data2.drop('dmrbin', axis=1)], axis=1)
data

In [ ]:
from scipy.stats import chi2_contingency
from statsmodels.stats.multitest import multipletests as FDR

for t in ['loop','summit']:
    stats = []
    for xx,yy,zz in data[[f'dmr{t}','dmrbin',t]].values:
        _,pv,*_ = chi2_contingency([[xx,zz-xx],[yy,tot_bin-yy]])
        stats.append(pv)
    
    data[f'{t}odd'] = data[f'dmr{t}'] / data[t] / data['dmrbin'] * tot_bin
    data[f'{t}fdr'] = FDR(stats, alpha=0.05, method='fdr_bh')[1]


In [ ]:
L1_meta = L1_meta.loc[L1_meta.index.isin(data.index)]

In [ ]:
rorder = data[['loopodd', 'summitodd']].mean(axis=1).sort_values().index[::-1]
fig, ax = plt.subplots(figsize=(2,6), dpi=300)
ax.imshow(np.log2(data.loc[rorder, ['loopodd', 'summitodd']]), 
          cmap='vlag', vmax=2, vmin=-2, rasterized=True, aspect='auto', interpolation='none')
ax.set_yticks(np.arange(data.shape[0]))
ax.set_yticklabels(rorder.map(L1_annot))
ax.set_xticks([0,1])
ax.set_xticklabels(['Loop','Summit'], rotation=90)
fig.tight_layout()
fig.savefig(f'{outdir}loop_summit_subtypedmr_enrich.pdf', transparent=True)


In [ ]:
data['dmrloop'] / data['dmrbin'] / data['loop'] * 249871